# Aplicação de Modelos de Previsão para Apoiar Decisões de Aprovisionamento visando a Redução de Custos

**Autora:** Izabella Santos  
**Orientadora:** Ana Paula Lopes  
**Instituição:** Instituto Superior de Contabilidade e Administração do Porto  
**Grau:** Mestrado em Logística  
**Ano:** 2026

***

## Descrição

Este notebook implementa o modelo de previsão de procura desenvolvido na dissertação de mestrado.  
São comparados cinco modelos aplicados à série temporal diária do *dataset* **DataCo Smart Supply Chain** (janeiro 2015 – setembro 2017):

| Modelo | Tipo |
|---|---|
| Naïve | Benchmark |
| SARIMA (1,0,1)(1,0,1)₇ | Estatístico |
| Holt-Winters (sazonalidade semanal) | Estatístico |
| Random Forest | Machine Learning |
| XGBoost | Machine Learning |

**Dataset:** Constante, F. J. N., Silva, F., & Pereira, A. C. (2019). *DataCo Smart Supply Chain for Big Data Analysis* [Dataset]. Mendeley Data. https://doi.org/10.17632/8gx2fvg2k6.5  
Após descarregar, colocar o ficheiro CSV na pasta `DataBase/`.


## 0. Instalação de Dependências

Executar apenas se as bibliotecas não estiverem instaladas.


In [ ]:
# !pip install pandas numpy matplotlib seaborn statsmodels scikit-learn xgboost


## 1. Importações e Configuração Gráfica


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from xgboost import XGBRegressor

sns.set_style('whitegrid')
colors = {
    'treino': '#426871', 'teste': '#D2B04C', 'naive': '#7B7B7B',
    'arima': '#A3623A', 'hw': '#C39B6A', 'rf': '#016E51', 'xgb': '#1A3C5E'
}
print('Bibliotecas carregadas com sucesso.')


## 2. Carregamento e Preparação dos Dados

O *dataset* DataCo cobre o período 2015–2018. Os dados de outubro a dezembro de 2017 são excluídos por apresentarem uma quebra abrupta de ~80% no volume, indicativa de dados incompletos.


In [ ]:
df = pd.read_csv("DataBase/DataCoSupplyChainDataset.csv", encoding="ISO-8859-1")
df['order_date'] = pd.to_datetime(df['order date (DateOrders)'])
df = df[(df['order_date'] >= '2015-01-01') & (df['order_date'] <= '2017-09-30')]
target = 'Order Item Quantity'
df_ts = df.groupby(df['order_date'].dt.date)[target].sum()
df_ts.index = pd.to_datetime(df_ts.index)
df_ts = df_ts.to_frame().asfreq('D', fill_value=0)
print(f'Série total : {len(df_ts)} dias')
print(f'Período     : {df_ts.index[0].date()} → {df_ts.index[-1].date()}')
print(f'Média diária: {df_ts[target].mean():.1f} unidades')


## 3. Divisão Treino / Teste (80% / 20%)


In [ ]:
train_size = int(len(df_ts) * 0.8)
train = df_ts[:train_size]
test  = df_ts[train_size:]
print(f'Treino : {len(train)} dias ({train.index[0].date()} → {train.index[-1].date()})')
print(f'Teste  : {len(test)} dias ({test.index[0].date()} → {test.index[-1].date()})')


## 4. Diagnóstico de Séries Temporais

### 4.1 Testes de Estacionaridade (ADF e KPSS)

- **Teste ADF**: H₀ = série não estacionária. Rejeitar se *p* ≤ 0,05
- **Teste KPSS**: H₀ = série estacionária. Rejeitar se *p* < 0,05


In [ ]:
adf_result  = adfuller(train[target], autolag='AIC')
kpss_result = kpss(train[target], regression='c', nlags='auto')
print(f'[ADF]  Estatística: {adf_result[0]:.4f} | p-value: {adf_result[1]:.4f}')
print(f'[KPSS] Estatística: {kpss_result[0]:.4f} | p-value: {kpss_result[1]:.4f}')
adf_stat  = adf_result[1]  <= 0.05
kpss_stat = kpss_result[1] >= 0.05
if adf_stat and kpss_stat:
    d_order = 0
    print('→ Ambos concordam: série estacionária. d = 0')
else:
    d_order = 1
    print('→ Decisão conservadora: d = 1')
print(f'→ d = {d_order}')


### 4.2 Funções de Autocorrelação (ACF e PACF)


In [ ]:
max_lags = min(30, len(train) // 2 - 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('ACF e PACF — Diagnóstico para Parametrização do SARIMA', fontsize=13, weight='bold')
plot_acf( train[target], lags=max_lags, ax=axes[0], color=colors['arima'])
plot_pacf(train[target], lags=max_lags, ax=axes[1], color=colors['treino'])
axes[0].set_title('ACF — Identifica ordem MA (q)')
axes[1].set_title('PACF — Identifica ordem AR (p)')
plt.tight_layout()
plt.show()


## 5. Função de Avaliação de Métricas


In [ ]:
def avaliar_modelo(model_name, real, pred):
    rmse = np.sqrt(mean_squared_error(real, pred))
    mape = mean_absolute_percentage_error(real, pred)
    print(f'--- {model_name} ---')
    print(f'  RMSE : {rmse:.2f}')
    print(f'  MAPE : {mape:.2%}')
    return rmse, mape


## 6. Modelo Naïve — Benchmark

O modelo Naïve projecta o último valor observado para todos os períodos futuros. Serve como referência mínima: qualquer modelo que não supere o Naïve não justifica a sua complexidade adicional (Hyndman & Athanasopoulos, 2021).


In [ ]:
naive_fc = pd.Series([train[target].iloc[-1]] * len(test), index=test.index)
naive_rmse, naive_mape = avaliar_modelo('Naïve (Benchmark)', test[target], naive_fc)


## 7. Modelo SARIMA

O SARIMA estende o ARIMA com componentes sazonais P, D, Q e período *s*.  
Configurado com sazonalidade semanal (*s* = 7). Ordem: **(1, 0, 1)(1, 0, 1)₇**


In [ ]:
sarima_model = SARIMAX(
    train[target], order=(1, d_order, 1),
    seasonal_order=(1, 0, 1, 7),
    enforce_stationarity=False, enforce_invertibility=False
).fit(disp=False)
arima_fc = sarima_model.forecast(steps=len(test))
arima_fc.index = test.index
arima_rmse, arima_mape = avaliar_modelo('SARIMA (1,0,1)(1,0,1)₇', test[target], arima_fc)


## 8. Modelo Holt-Winters

Suavização exponencial tripla com tendência e sazonalidade aditivas. *seasonal_periods* = 7 (ciclo semanal).


In [ ]:
hw_model = ExponentialSmoothing(
    train[target], trend='add', seasonal='add',
    seasonal_periods=7, initialization_method='estimated'
).fit()
hw_fc = hw_model.forecast(len(test))
hw_fc.index = test.index
hw_rmse, hw_mape = avaliar_modelo('Holt-Winters (sazonal semanal)', test[target], hw_fc)


## 9. Feature Engineering para Modelos de Machine Learning

| Feature | Descrição |
|---|---|
| `day_num` | Tendência global |
| `day_of_week` | Sazonalidade semanal |
| `day_of_month` | Posição no mês |
| `month_of_year` | Sazonalidade anual |
| `lag_1` | Procura do dia anterior |
| `lag_7` | Mesmo dia na semana anterior |
| `lag_14` | Mesmo dia há duas semanas |
| `rolling_mean_7` | Média móvel 7 dias |
| `rolling_mean_14` | Média móvel 14 dias |


In [ ]:
df_feat = df_ts.copy()
df_feat['day_num']         = np.arange(len(df_feat))
df_feat['day_of_week']     = df_feat.index.dayofweek
df_feat['day_of_month']    = df_feat.index.day
df_feat['month_of_year']   = df_feat.index.month
df_feat['lag_1']           = df_feat[target].shift(1)
df_feat['lag_7']           = df_feat[target].shift(7)
df_feat['lag_14']          = df_feat[target].shift(14)
df_feat['rolling_mean_7']  = df_feat[target].rolling(7).mean()
df_feat['rolling_mean_14'] = df_feat[target].rolling(14).mean()
df_feat = df_feat.dropna()
features = ['day_num','day_of_week','day_of_month','month_of_year',
            'lag_1','lag_7','lag_14','rolling_mean_7','rolling_mean_14']
train_feat = df_feat[df_feat.index < test.index[0]]
test_feat  = df_feat[df_feat.index >= test.index[0]]
print(f'Treino ML: {len(train_feat)} obs. | Teste ML: {len(test_feat)} obs.')


## 10. Modelo Random Forest

Combina 200 árvores de decisão independentes, minimizando a variância do erro de previsão (Breiman, 2001).


In [ ]:
rf_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(train_feat[features], train_feat[target])
rf_fc = pd.Series(rf_model.predict(test_feat[features]), index=test_feat.index)
rf_rmse, rf_mape = avaliar_modelo('Random Forest', test[target], rf_fc)

feat_imp = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
feat_imp.plot(kind='barh', ax=ax, color=colors['rf'])
ax.set_title('Random Forest — Importância das Features', fontsize=13, weight='bold')
ax.set_xlabel('Importância relativa')
plt.tight_layout()
plt.show()


## 11. Modelo XGBoost

Constrói árvores sequencialmente, corrigindo os erros residuais da árvore anterior (Chen & Guestrin, 2016).


In [ ]:
xgb_model = XGBRegressor(
    n_estimators=200, learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0
)
xgb_model.fit(train_feat[features], train_feat[target])
xgb_fc = pd.Series(xgb_model.predict(test_feat[features]), index=test_feat.index)
xgb_rmse, xgb_mape = avaliar_modelo('XGBoost', test[target], xgb_fc)

xgb_feat_imp = pd.Series(xgb_model.feature_importances_, index=features).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
xgb_feat_imp.plot(kind='barh', ax=ax, color=colors['xgb'])
ax.set_title('XGBoost — Importância das Features', fontsize=13, weight='bold')
ax.set_xlabel('Importância relativa')
plt.tight_layout()
plt.show()


## 12. Tabela Comparativa de Resultados


In [ ]:
resultados = pd.DataFrame({
    'Modelo': ['Naïve (Benchmark)', 'SARIMA', 'Holt-Winters', 'Random Forest', 'XGBoost'],
    'Tipo':   ['Benchmark', 'Estatístico', 'Estatístico', 'Machine Learning', 'Machine Learning'],
    'RMSE':   [naive_rmse, arima_rmse, hw_rmse, rf_rmse, xgb_rmse],
    'MAPE':   [naive_mape, arima_mape, hw_mape, rf_mape, xgb_mape]
}).sort_values('MAPE').reset_index(drop=True)
resultados.index = resultados.index + 1
resultados.index.name = 'Rank'
resultados['MAPE (%)'] = resultados['MAPE'].map('{:.2%}'.format)
resultados['RMSE'] = resultados['RMSE'].map('{:.2f}'.format)
print(resultados[['Modelo','Tipo','RMSE','MAPE (%)']].to_string())


## 13. Gráfico Comparativo — RMSE e MAPE


In [ ]:
modelos_nomes = ['Random Forest', 'XGBoost', 'Holt-Winters', 'SARIMA', 'Naïve']
rmse_vals = [rf_rmse, xgb_rmse, hw_rmse, arima_rmse, naive_rmse]
mape_vals = [rf_mape*100, xgb_mape*100, hw_mape*100, arima_mape*100, naive_mape*100]

fig, ax1 = plt.subplots(figsize=(13, 6.5))
fig.patch.set_facecolor('white')
ax1.set_facecolor('#FAFAFA')
x = np.arange(len(modelos_nomes))
width = 0.35

bars1 = ax1.bar(x - width/2, rmse_vals, width, color='#426871', alpha=0.92,
                edgecolor='white', linewidth=0.5, label='RMSE')
ax2 = ax1.twinx()
bars2 = ax2.bar(x + width/2, mape_vals, width, color='#D2B04C', alpha=0.88,
                edgecolor='white', linewidth=0.5, label='MAPE (%)')

for bar, val in zip(bars1, rmse_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
             f'{val:.2f}', ha='center', va='bottom', fontsize=9,
             color='#426871', fontweight='bold')
for bar, val in zip(bars2, mape_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{val:.2f}%', ha='center', va='bottom', fontsize=9,
             color='#8a5800', fontweight='bold')

ax1.set_ylabel('RMSE', fontsize=12, color='#426871', fontweight='bold')
ax2.set_ylabel('MAPE (%)', fontsize=12, color='#8a5800', fontweight='bold')
ax1.tick_params(axis='y', labelcolor='#426871')
ax2.tick_params(axis='y', labelcolor='#8a5800')
ax1.set_xticks(x)
ax1.set_xticklabels(modelos_nomes, fontsize=10.5)
ax1.set_ylim(0, 57)
ax2.set_ylim(0, 14)
ax1.yaxis.grid(True, alpha=0.35, linestyle='--', color='#cccccc')
ax1.set_axisbelow(True)
ax1.spines[['top', 'right']].set_visible(False)
ax2.spines[['top']].set_visible(False)
ax1.set_title('Comparação de Modelos — RMSE e MAPE (Dados Diários)',
              fontsize=13, fontweight='bold', pad=18)

import matplotlib.patches as mpatches
patch_rmse = mpatches.Patch(color='#426871', alpha=0.92, label='RMSE')
patch_mape = mpatches.Patch(color='#D2B04C', alpha=0.88, label='MAPE (%)')
fig.legend(handles=[patch_rmse, patch_mape], loc='upper right',
           bbox_to_anchor=(0.98, 0.98), frameon=True, framealpha=0.95,
           edgecolor='#cccccc', fontsize=10.5)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig('comparacao_modelos.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()


## 14. Gráficos de Previsão por Modelo

Cada gráfico mostra os últimos 60 dias do período de treino e o período de teste completo (201 dias).


In [ ]:
def plot_forecast(train, test, forecast, model_name, color):
    fig, ax = plt.subplots(figsize=(14, 5))
    train_tail = train.tail(60)
    ax.plot(train_tail.index, train_tail[target], label='Histórico (Treino)',
            color=colors['treino'], linewidth=1.5)
    ax.plot(test.index, test[target], label='Real (Teste)',
            color=colors['teste'], linewidth=1.5)
    ax.plot(test.index, forecast, label=f'Previsão {model_name}',
            linestyle='--', color=color, linewidth=2)
    ax.set_title(f'Forecasting de Procura Diária (2015–2017) — {model_name}',
                 fontsize=13, weight='bold')
    ax.set_xlabel('Data')
    ax.set_ylabel('Quantidade')
    ax.legend(loc='upper left')
    plt.tight_layout()
    plt.show()


In [ ]:
plot_forecast(train, test, naive_fc, 'Naïve (Benchmark)', colors['naive'])


In [ ]:
plot_forecast(train, test, arima_fc, 'SARIMA', colors['arima'])


In [ ]:
plot_forecast(train, test, hw_fc, 'Holt-Winters', colors['hw'])


In [ ]:
plot_forecast(train, test, rf_fc, 'Random Forest', colors['rf'])


In [ ]:
plot_forecast(train, test, xgb_fc, 'XGBoost', colors['xgb'])


## 15. Análise por Categoria — 7 Classes A (Curva de Pareto)

A análise de Pareto sobre as categorias de produtos revelou que **7 categorias representam ~81% do volume total** de encomendas. Para avaliar se a vantagem dos modelos de Machine Learning se mantém a este nível de granularidade, os cinco modelos foram aplicados individualmente a cada uma destas categorias, com os mesmos hiperparâmetros da série total.

| Categoria | Vol. (% total) | Média diária |
|---|---|---|
| Cleats | 19,6% | 73,3 u/dia |
| Women's Apparel | 16,8% | 62,6 u/dia |
| Indoor/Outdoor Games | 15,4% | 57,5 u/dia |
| Cardio Equipment | 10,0% | 37,4 u/dia |
| Shop By Sport | 8,7% | 32,6 u/dia |
| Men's Footwear | 5,9% | 22,1 u/dia |
| Fishing | 4,6% | 17,2 u/dia |


In [ ]:
# Função auxiliar de feature engineering por série
def df_feat_func(ts_frame, target_col):
    d = ts_frame.copy()
    d['day_num']         = np.arange(len(d))
    d['day_of_week']     = d.index.dayofweek
    d['day_of_month']    = d.index.day
    d['month_of_year']   = d.index.month
    d['lag_1']           = d[target_col].shift(1)
    d['lag_7']           = d[target_col].shift(7)
    d['lag_14']          = d[target_col].shift(14)
    d['rolling_mean_7']  = d[target_col].rolling(7).mean()
    d['rolling_mean_14'] = d[target_col].rolling(14).mean()
    return d.dropna()

# Função que corre os 5 modelos em qualquer série temporal
def run_all_models(series_df, target_col, d_param=0):
    n = len(series_df); split = int(n * 0.8)
    tr = series_df.iloc[:split]; te = series_df.iloc[split:]
    res = {}
    # Naïve
    naive_p = pd.Series([tr[target_col].iloc[-1]] * len(te), index=te.index)
    res['Naïve'] = {
        'RMSE': round(np.sqrt(mean_squared_error(te[target_col], naive_p)), 2),
        'MAPE': round(mean_absolute_percentage_error(te[target_col], naive_p) * 100, 2)
    }
    # SARIMA
    try:
        m = SARIMAX(tr[target_col], order=(1,d_param,1), seasonal_order=(1,0,1,7),
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=200)
        p = np.maximum(pd.Series(m.forecast(len(te)).values, index=te.index), 0)
        res['SARIMA'] = {'RMSE': round(np.sqrt(mean_squared_error(te[target_col], p)), 2),
                         'MAPE': round(mean_absolute_percentage_error(te[target_col], p)*100, 2)}
    except:
        res['SARIMA'] = {'RMSE': None, 'MAPE': None}
    # Holt-Winters
    try:
        hw = ExponentialSmoothing(tr[target_col], trend='add', seasonal='add',
                                  seasonal_periods=7, initialization_method='estimated').fit()
        p = np.maximum(pd.Series(hw.forecast(len(te)).values, index=te.index), 0)
        res['Holt-Winters'] = {'RMSE': round(np.sqrt(mean_squared_error(te[target_col], p)), 2),
                               'MAPE': round(mean_absolute_percentage_error(te[target_col], p)*100, 2)}
    except:
        res['Holt-Winters'] = {'RMSE': None, 'MAPE': None}
    # ML
    df_f = df_feat_func(series_df, target_col)
    fs = split - 14; dtr_f = df_f.iloc[:fs]; dte_f = df_f.iloc[fs:]
    if len(dtr_f) >= 50:
        real = dte_f[target_col].values
        rf2 = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
        rf2.fit(dtr_f[features], dtr_f[target_col])
        pr = np.maximum(rf2.predict(dte_f[features]), 0)
        res['Random Forest'] = {'RMSE': round(np.sqrt(mean_squared_error(real, pr)), 2),
                                'MAPE': round(mean_absolute_percentage_error(real, pr)*100, 2)}
        xgb2 = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4,
                             subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0)
        xgb2.fit(dtr_f[features], dtr_f[target_col])
        px = np.maximum(xgb2.predict(dte_f[features]), 0)
        res['XGBoost'] = {'RMSE': round(np.sqrt(mean_squared_error(real, px)), 2),
                          'MAPE': round(mean_absolute_percentage_error(real, px)*100, 2)}
    else:
        res['Random Forest'] = res['XGBoost'] = {'RMSE': None, 'MAPE': None}
    return res

CATEGORIAS_CLASSE_A = ['Cleats', "Women's Apparel", 'Indoor/Outdoor Games',
                        'Cardio Equipment', 'Shop By Sport', "Men's Footwear", 'Fishing']
print('Funções definidas.')


In [ ]:
# Correr os 5 modelos nas 7 categorias Classe A
resultados_cats = {
    'TOTAL (agregado)': {
        'Naïve':         {'RMSE': 47.50, 'MAPE': 11.80},
        'SARIMA':        {'RMSE': 43.73, 'MAPE': 10.69},
        'Holt-Winters':  {'RMSE': 43.63, 'MAPE': 10.64},
        'Random Forest': {'RMSE': 35.87, 'MAPE':  8.24},
        'XGBoost':       {'RMSE': 38.35, 'MAPE':  8.84},
    }
}

for cat in CATEGORIAS_CLASSE_A:
    cat_df = df[df['Category Name'] == cat]
    serie  = cat_df.groupby(cat_df['order_date'].dt.date)[target].sum()
    serie.index = pd.to_datetime(serie.index)
    sf = serie.to_frame().asfreq('D', fill_value=0)
    resultados_cats[cat] = run_all_models(sf, target, d_param=d_order)
    r = resultados_cats[cat]
    print(f"{cat:<30} Naïve {r['Naïve']['MAPE']}%  RF {r['Random Forest']['MAPE']}%  XGB {r['XGBoost']['MAPE']}%")


## 16. Tabela Resumo — MAPE (%) por Modelo e Categoria


In [ ]:
MODELOS_CAT = ['Naïve', 'SARIMA', 'Holt-Winters', 'Random Forest', 'XGBoost']
rows = []
for cat, res in resultados_cats.items():
    row = {'Categoria': cat}
    for m in MODELOS_CAT:
        row[m] = f"{res[m]['MAPE']}%" if res[m]['MAPE'] else 'N/A'
    rows.append(row)
df_tab = pd.DataFrame(rows).set_index('Categoria')
print(df_tab.to_string())


## 17. Gráfico Comparativo — MAPE por Modelo e Categoria

Dois padrões destacam-se:
- **Categorias voláteis** (Cardio Equipment, Indoor/Outdoor Games): o ML reduz o MAPE do Naïve em 35–47%
- **Categorias estáveis** (Men's Footwear, Cleats): todos os modelos convergem — o Naïve é difícil de superar


In [ ]:
cat_labels = list(resultados_cats.keys())
x = np.arange(len(cat_labels)); w = 0.15
cores_mod = [colors['naive'], colors['arima'], colors['hw'], colors['rf'], colors['xgb']]

fig, ax = plt.subplots(figsize=(16, 7))
fig.patch.set_facecolor('#F8F8F8'); ax.set_facecolor('#FAFAFA')

for i, (mod, cor) in enumerate(zip(MODELOS_CAT, cores_mod)):
    vals = [resultados_cats[c][mod]['MAPE'] or 0 for c in cat_labels]
    bars = ax.bar(x + i*w, vals, w, label=mod, color=cor, alpha=0.88,
                  edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, vals):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
                    f'{val:.1f}%', ha='center', va='bottom',
                    fontsize=6.5, color='#333333', fontweight='bold')

ax.set_xticks(x + w*2)
ax.set_xticklabels(cat_labels, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('MAPE (%)'); ax.set_ylim(0, ax.get_ylim()[1] * 1.12)
ax.set_title('Comparação de MAPE por Modelo — Total e 7 Categorias Classe A',
             fontsize=13, weight='bold', pad=14)
ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
ax.yaxis.grid(True, alpha=0.4, linestyle='--'); ax.set_axisbelow(True)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('comparacao_mape_categorias.png', dpi=300, bbox_inches='tight', facecolor='#F8F8F8')
plt.show()


## 18. Gráficos de Previsão — Random Forest por Categoria

O Random Forest foi o melhor modelo global (MAPE 8,24%). Os gráficos mostram o seu desempenho em cada uma das 7 categorias Classe A.


### Cleats — MAPE RF: 19,29%


In [ ]:
cat_name = 'Cleats'
cat_df_plot = df[df['Category Name'] == cat_name]
serie_plot = cat_df_plot.groupby(cat_df_plot['order_date'].dt.date)[target].sum()
serie_plot.index = pd.to_datetime(serie_plot.index)
sf_plot = serie_plot.to_frame().asfreq('D', fill_value=0)
n_plot = len(sf_plot); split_plot = int(n_plot * 0.8)
tr_plot = sf_plot.iloc[:split_plot]; te_plot = sf_plot.iloc[split_plot:]
df_f_plot = df_feat_func(sf_plot, target)
fs_plot = split_plot - 14
dtr_plot = df_f_plot.iloc[:fs_plot]; dte_plot = df_f_plot.iloc[fs_plot:]
rf_cat = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_cat.fit(dtr_plot[features], dtr_plot[target])
rf_cat_fc = pd.Series(
    np.maximum(rf_cat.predict(dte_plot[features]), 0),
    index=dte_plot.index
)
plot_forecast(tr_plot, te_plot, rf_cat_fc, f'Random Forest — {cat_name}', colors['rf'])


### Women's Apparel — MAPE RF: 26,01%


In [ ]:
cat_name = 'Women's Apparel'
cat_df_plot = df[df['Category Name'] == cat_name]
serie_plot = cat_df_plot.groupby(cat_df_plot['order_date'].dt.date)[target].sum()
serie_plot.index = pd.to_datetime(serie_plot.index)
sf_plot = serie_plot.to_frame().asfreq('D', fill_value=0)
n_plot = len(sf_plot); split_plot = int(n_plot * 0.8)
tr_plot = sf_plot.iloc[:split_plot]; te_plot = sf_plot.iloc[split_plot:]
df_f_plot = df_feat_func(sf_plot, target)
fs_plot = split_plot - 14
dtr_plot = df_f_plot.iloc[:fs_plot]; dte_plot = df_f_plot.iloc[fs_plot:]
rf_cat = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_cat.fit(dtr_plot[features], dtr_plot[target])
rf_cat_fc = pd.Series(
    np.maximum(rf_cat.predict(dte_plot[features]), 0),
    index=dte_plot.index
)
plot_forecast(tr_plot, te_plot, rf_cat_fc, f'Random Forest — {cat_name}', colors['rf'])


### Indoor/Outdoor Games — MAPE RF: 23,71%


In [ ]:
cat_name = 'Indoor/Outdoor Games'
cat_df_plot = df[df['Category Name'] == cat_name]
serie_plot = cat_df_plot.groupby(cat_df_plot['order_date'].dt.date)[target].sum()
serie_plot.index = pd.to_datetime(serie_plot.index)
sf_plot = serie_plot.to_frame().asfreq('D', fill_value=0)
n_plot = len(sf_plot); split_plot = int(n_plot * 0.8)
tr_plot = sf_plot.iloc[:split_plot]; te_plot = sf_plot.iloc[split_plot:]
df_f_plot = df_feat_func(sf_plot, target)
fs_plot = split_plot - 14
dtr_plot = df_f_plot.iloc[:fs_plot]; dte_plot = df_f_plot.iloc[fs_plot:]
rf_cat = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_cat.fit(dtr_plot[features], dtr_plot[target])
rf_cat_fc = pd.Series(
    np.maximum(rf_cat.predict(dte_plot[features]), 0),
    index=dte_plot.index
)
plot_forecast(tr_plot, te_plot, rf_cat_fc, f'Random Forest — {cat_name}', colors['rf'])


### Cardio Equipment — MAPE RF: 30,84%


In [ ]:
cat_name = 'Cardio Equipment'
cat_df_plot = df[df['Category Name'] == cat_name]
serie_plot = cat_df_plot.groupby(cat_df_plot['order_date'].dt.date)[target].sum()
serie_plot.index = pd.to_datetime(serie_plot.index)
sf_plot = serie_plot.to_frame().asfreq('D', fill_value=0)
n_plot = len(sf_plot); split_plot = int(n_plot * 0.8)
tr_plot = sf_plot.iloc[:split_plot]; te_plot = sf_plot.iloc[split_plot:]
df_f_plot = df_feat_func(sf_plot, target)
fs_plot = split_plot - 14
dtr_plot = df_f_plot.iloc[:fs_plot]; dte_plot = df_f_plot.iloc[fs_plot:]
rf_cat = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_cat.fit(dtr_plot[features], dtr_plot[target])
rf_cat_fc = pd.Series(
    np.maximum(rf_cat.predict(dte_plot[features]), 0),
    index=dte_plot.index
)
plot_forecast(tr_plot, te_plot, rf_cat_fc, f'Random Forest — {cat_name}', colors['rf'])


### Shop By Sport — MAPE RF: 31,34%


In [ ]:
cat_name = 'Shop By Sport'
cat_df_plot = df[df['Category Name'] == cat_name]
serie_plot = cat_df_plot.groupby(cat_df_plot['order_date'].dt.date)[target].sum()
serie_plot.index = pd.to_datetime(serie_plot.index)
sf_plot = serie_plot.to_frame().asfreq('D', fill_value=0)
n_plot = len(sf_plot); split_plot = int(n_plot * 0.8)
tr_plot = sf_plot.iloc[:split_plot]; te_plot = sf_plot.iloc[split_plot:]
df_f_plot = df_feat_func(sf_plot, target)
fs_plot = split_plot - 14
dtr_plot = df_f_plot.iloc[:fs_plot]; dte_plot = df_f_plot.iloc[fs_plot:]
rf_cat = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_cat.fit(dtr_plot[features], dtr_plot[target])
rf_cat_fc = pd.Series(
    np.maximum(rf_cat.predict(dte_plot[features]), 0),
    index=dte_plot.index
)
plot_forecast(tr_plot, te_plot, rf_cat_fc, f'Random Forest — {cat_name}', colors['rf'])


### Men's Footwear — MAPE RF: 18,20%


In [ ]:
cat_name = 'Men's Footwear'
cat_df_plot = df[df['Category Name'] == cat_name]
serie_plot = cat_df_plot.groupby(cat_df_plot['order_date'].dt.date)[target].sum()
serie_plot.index = pd.to_datetime(serie_plot.index)
sf_plot = serie_plot.to_frame().asfreq('D', fill_value=0)
n_plot = len(sf_plot); split_plot = int(n_plot * 0.8)
tr_plot = sf_plot.iloc[:split_plot]; te_plot = sf_plot.iloc[split_plot:]
df_f_plot = df_feat_func(sf_plot, target)
fs_plot = split_plot - 14
dtr_plot = df_f_plot.iloc[:fs_plot]; dte_plot = df_f_plot.iloc[fs_plot:]
rf_cat = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_cat.fit(dtr_plot[features], dtr_plot[target])
rf_cat_fc = pd.Series(
    np.maximum(rf_cat.predict(dte_plot[features]), 0),
    index=dte_plot.index
)
plot_forecast(tr_plot, te_plot, rf_cat_fc, f'Random Forest — {cat_name}', colors['rf'])


### Fishing — MAPE RF: 20,25%


In [ ]:
cat_name = 'Fishing'
cat_df_plot = df[df['Category Name'] == cat_name]
serie_plot = cat_df_plot.groupby(cat_df_plot['order_date'].dt.date)[target].sum()
serie_plot.index = pd.to_datetime(serie_plot.index)
sf_plot = serie_plot.to_frame().asfreq('D', fill_value=0)
n_plot = len(sf_plot); split_plot = int(n_plot * 0.8)
tr_plot = sf_plot.iloc[:split_plot]; te_plot = sf_plot.iloc[split_plot:]
df_f_plot = df_feat_func(sf_plot, target)
fs_plot = split_plot - 14
dtr_plot = df_f_plot.iloc[:fs_plot]; dte_plot = df_f_plot.iloc[fs_plot:]
rf_cat = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_cat.fit(dtr_plot[features], dtr_plot[target])
rf_cat_fc = pd.Series(
    np.maximum(rf_cat.predict(dte_plot[features]), 0),
    index=dte_plot.index
)
plot_forecast(tr_plot, te_plot, rf_cat_fc, f'Random Forest — {cat_name}', colors['rf'])


## 19. Discussão — Análise por Categoria

### Padrões identificados

**Categorias onde o ML tem vantagem expressiva:**
| Categoria | MAPE Naïve | MAPE RF | Redução |
|---|---|---|---|
| Indoor/Outdoor Games | 45,04% | 23,71% | **−47%** |
| Cardio Equipment | 47,61% | 30,84% | **−35%** |
| Women's Apparel | 31,07% | 26,01% | −16% |
| Fishing | 24,50% | 20,25% | −17% |

**Categorias onde os modelos convergem:**
| Categoria | MAPE Naïve | MAPE RF | Diferença |
|---|---|---|---|
| Men's Footwear | 18,18% | 18,20% | ≈0 |
| Cleats | 20,04% | 19,29% | −0,75 pp |

**Caso especial — Shop By Sport:** o RF (31,34%) não supera o Naïve (30,63%). A elevada volatilidade pontual desta categoria não tem padrão aprendível com o histórico disponível — ausência de variáveis exógenas (promoções, eventos desportivos) (Januschowski et al., 2020).

### Implicação para o aprovisionamento
A análise confirma que a vantagem do ML é **heterogénea e proporcional à volatilidade da série**. Para o gestor de compras, isto traduz-se numa estratégia diferenciada:
- **Categorias voláteis**: investir em modelos ML — o ganho de precisão é substancial e tem impacto directo no stock de segurança
- **Categorias estáveis**: modelos simples (Naïve, médias móveis) são suficientes — a complexidade adicional não se justifica

Este resultado é consistente com Makridakis et al. (2018, 2022): a vantagem do ML sobre os métodos estatísticos manifesta-se principalmente em séries com estrutura não linear complexa e elevada variabilidade residual.


## Referências

- Breiman, L. (2001). Random Forests. *Machine Learning*, 45(1), 5–32.
- Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *Proceedings of the 22nd ACM SIGKDD*, 785–794. https://doi.org/10.1145/2939672.2939785
- Chopra, S., & Meindl, P. (2016). *Supply chain management* (6.ª ed.). Pearson.
- Christopher, M. (2016). *Logistics & supply chain management* (5.ª ed.). Pearson.
- Constante, F. J. N., Silva, F., & Pereira, A. C. (2019). *DataCo Smart Supply Chain for Big Data Analysis* [Dataset]. Mendeley Data. https://doi.org/10.17632/8gx2fvg2k6.5
- Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and practice* (3.ª ed.). OTexts. https://otexts.com/fpp3/
- Januschowski, T., Gasthaus, J., Park, Y., Salinas, D., Flunkert, V., Seeger, M., & Smola, A. J. (2020). Criteria for classifying forecasting methods. *International Journal of Forecasting*, 36(1), 167–177.
- Makridakis, S., Spiliotis, E., & Assimakopoulos, V. (2018). Statistical and Machine Learning forecasting methods. *PLoS ONE*, 13(3), e0194889. https://doi.org/10.1371/journal.pone.0194889
- Makridakis, S., Spiliotis, E., & Assimakopoulos, V. (2022). M5 accuracy competition: Results, findings, and conclusions. *International Journal of Forecasting*, 38(4), 1346–1364.
- Silver, E. A., Pyke, D. F., & Peterson, R. (1998). *Inventory management and production planning and scheduling* (3.ª ed.). Wiley.
